# Q1: 960. Binning continuous variables




Turn a continuous feature into discrete bucket IDs using equal-width binning.

$$w = \frac{\max(x) - \min(x)}{\text{num\_bins}}$$

$$\text{bin}(x_i) = \left\lfloor \frac{x_i - \min(x)}{w} \right\rfloor$$

Clamp max value to last bin: $\text{bin}(x_{max}) = \text{num\_bins} - 1$

**Constraints**
- NumPy and Python built-ins only (no pandas, no scikit-learn)
- Return both `bin_ids` (integer) and `one_hot` encoding

**Input**
```
x        = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
num_bins = 3
```
**Expected Output**
```
bin_ids  = [0, 0, 0, 1, 1, 1, 2, 2, 2, 2]
one_hot  = [[1,0,0], [1,0,0], [1,0,0],
            [0,1,0], [0,1,0], [0,1,0],
            [0,0,1], [0,0,1], [0,0,1], [0,0,1]]
```

In [ ]:
import numpy as np

def bin_continuous_feature(x, num_bins):
    """
    Discretizes a continuous feature into equal-width bins.

    The bins span from min(x) to max(x), split into `num_bins`
    equal intervals. Each value is assigned a bin index in
    [0, num_bins - 1].

    Args:
        x (np.ndarray): 1D array of continuous values.
        num_bins (int): Number of bins to create.

    Returns:
        tuple[np.ndarray, np.ndarray]:
            - bin_ids: array of shape (len(x),), where each entry
              is the assigned bin index.
            - one_hot: array of shape (len(x), num_bins), each entry is a
              one-hot encoded vector.
    """
    # Ensure input is a numpy array
    x = np.array(x)

    # Compute range and bin width
    x_min = np.min(x)
    x_max = np.max(x)

    # Avoid division by zero if all values are the same
    if x_max == x_min:
         width = 1.0 # arbitrary, since resulting bin will be 0 anyway if we handle correctly
    else:
         width = (x_max - x_min) / num_bins

    if x_max == x_min:
         # create a array of zeros with the same shape and dtype as the input array.
         bin_ids = np.zeros_like(x, dtype=int)
    else:
         # Assign bin indices
         # Note: We want floor((val - min) / width)
         bin_ids = ((x - x_min) / width).astype(int)
         print(bin_ids)
         # Clamp indices to [0, num_bins - 1] to handle the max value case
         bin_ids = np.clip(bin_ids, 0, num_bins - 1)    

    # Build one-hot encoding using advanced indexing
    one_hot = np.zeros((len(x), num_bins), dtype=int)
    print(one_hot)
    print(np.arange(len(x))) 
    one_hot[np.arange(len(x)), bin_ids] = 1 # this indexing I sould know

    return bin_ids, one_hot


# Example Usage
x = [1,2,2.9,3,4]
num_bins = 3
result = bin_continuous_feature(x, num_bins)
result

[0 1 1 2 3]
[[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]]
[0 1 2 3 4]


(array([0, 1, 1, 2, 2]),
 array([[1, 0, 0],
        [0, 1, 0],
        [0, 1, 0],
        [0, 0, 1],
        [0, 0, 1]]))

# Q2. 1387. Cumulative sum features




Build cumulative-sum (running total) features — a useful feature-engineering trick for time-ordered data. Compute per-group running totals so each row uses information from earlier rows only.

$$\text{cumsum}_i = \sum_{j=1}^{i} x_j$$

**Constraints**
- No pandas — NumPy and built-in Python only
- Input is already in time order — do not sort
- Cumulative sum resets per group
- Single pass `O(n)` expected

**Input**
```
values = [10.0, 5.0, 2.0, 7.0, 1.0]
groups = ["A",  "A", "B", "A", "B"]
```
**Expected Output**
```
[10.0, 15.0, 2.0, 22.0, 3.0]

index 0: A → 10
index 1: A → 10 + 5  = 15
index 2: B → 2              ← B resets
index 3: A → 15 + 7  = 22   ← A continues
index 4: B → 2  + 1  = 3    ← B continues
```

In [ ]:
import numpy as np

def cumulative_sum_features(values, groups):
    """
    Computes per-group cumulative sum (running total) features.

    Args:
        values (np.ndarray): Numeric values in time order.
        groups (np.ndarray): Group id for each value (same length as values).

    Returns:
        np.ndarray: A list where output[i] is the cumulative sum of `values`
        within `groups[i]` up to and including index i.
    """
    # Track running totals per group
    # Using a dictionary for O(N) single pass time complexity
    totals = {}
    output = [0.0] * len(values)

    # Iterate directly; if inputs are numpy arrays, this works fine
    for i, (value, group) in enumerate(zip(values, groups)):
        # Update running total for the group
        total = totals.get(group, 0.0) + value
        totals[group] = total
        output[i] = total

    return np.array(output)

result = cumulative_sum_features(values, groups)
result

# Q3. 413. Detecting duplicate features




Detect duplicate features — two columns are duplicates if they are exactly equal for every row.

$$x \equiv y \iff \forall i \in \{1, \dots, n\},\ x_i = y_i$$

**Constraints**
- No pandas — NumPy and built-in Python only
- Compare columns exactly — no string conversion
- Return pairs `(i < j)` in scan order
- Return feature names, not indices

**Input**
```
X = [[1.0, 5.0, 1.0],
     [2.0, 6.0, 2.0],
     [3.0, 7.0, 3.0]]

feature_names = ["a", "b", "c"]
```
**Expected Output**
```
[["a", "c"]]   ← col "a" and col "c" are identical
```

In [ ]:
import numpy as np

def find_duplicate_features(X, feature_names):
    """
    Finds duplicate features (identical columns) in a dataset.

    Args:
        X (np.ndarray): Table of shape (n_rows, n_features).
        feature_names (np.ndarray): Names of the features (length = n_features).

    Returns:
        np.ndarray: A 2D array of shape (N_pairs, 2) where each row is [a, b]
        meaning feature b is a duplicate of feature a.
        Return pairs in the order you discover them by scanning
        feature pairs (i < j).
    """
    n_features = X.shape[1]
    duplicates = []

    # Compare each pair of columns in order (i < j)
    for i in range(n_features):
        col_i = X[:, i]
        for j in range(i + 1, n_features):
            col_j = X[:, j]
            # Check if the two columns are exactly identical
            if np.array_equal(col_i, col_j):
                duplicates.append([feature_names[i], feature_names[j]])

    if not duplicates:
        return np.empty((0, 2))

    return np.array(duplicates)

result = find_duplicate_features(X, feature_names)
result

# Q4. 794. Feature clipping




Clip each numeric feature column using per-feature lower and upper bounds — reduces the impact of extreme values.

$$x'_{i,j} = \min(\max(x_{i,j},\ \text{lower}_j),\ \text{upper}_j)$$

**Constraints**
- No pandas — NumPy and built-in Python only
- Do not modify `X` in place — return a new array
- Treat each column independently using its own bounds

**Input**
```
X     = [[ 1.0, 100.0],
         [-5.0,  50.0],
         [ 3.0, 1000.0]]

lower = [0.0,  60.0]
upper = [2.0, 200.0]
```
**Expected Output**
```
[[ 1.0, 100.0],   ← 1.0 in [0,2], 100.0 in [60,200]
 [ 0.0,  60.0],   ← -5 clipped to 0,  50 clipped to 60
 [ 2.0, 200.0]]   ← 3  clipped to 2, 1000 clipped to 200
```

In [ ]:
import numpy as np

def clip_features(X, lower, upper):
    """
    Clips each feature column of X to a per-feature [lower_j, upper_j] range.

    Args:
        X (np.ndarray): 2D data matrix of shape (n_samples, n_features).
        lower (np.ndarray): Per-feature lower bounds of length n_features.
        upper (np.ndarray): Per-feature upper bounds of length n_features.

    Returns:
        np.ndarray: A new 2D array with the same shape as X, where each value
        is clipped to its feature's bounds.
    """
    # Element-wise clip (broadcasting lower/upper per column if needed, 
    # but np.clip does the right thing if shapes align or broadcast correctly)
    # Here lower and upper are shape (n_features,). X is (n_samples, n_features).
    # We clipping per column.
    return np.clip(X, lower, upper)

result = clip_features(X, lower, upper)
result

# Q5. 402. Feature selection variance threshold




Selecting useful features is a key step in feature engineering, and one simple baseline is to drop features that barely change across samples. In this task, you’ll implement variance-threshold feature selection for a tabular dataset represented as a NumPy array.

Requirements
Implement the function

python
Copy
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
⌄
⌄
import numpy as np

def variance_threshold_select(X, threshold):
    """
    Removes low-variance features (columns) from a dataset.

    Variance is defined as:
    $$
    \\mathrm{Var}(x) = \\frac{1}{n}\\sum_{i=1}^{n}(x_i - \\mu)^2
    $$
    where $\\mu$ is the mean of the feature values.

    Args:
        X (np.ndarray): 2D dataset of shape (n_samples, n_features).
        threshold (float): Keep only features with variance strictly greater than this value.

    Returns:
        np.ndarray: The transformed dataset with only selected columns, shape (n_samples, n_selected_features).
    """
    # Your solution below
Rules:

Compute the variance per feature (per column) across all samples.
Keep a feature only if its variance is strictly greater than threshold.
Return the filtered dataset as a NumPy array (same row order, fewer columns).
Do not use prebuilt feature selection utilities (e.g., sklearn.feature_selection.VarianceThreshold).
Key considerations: vectorize with NumPy, treat columns independently, preserve original column order.
Example
python
Copy
1
2
3
4
5
6
7
8
⌄
X = np.array([
    [1.0, 10.0, 0.0],
    [1.0, 11.0, 0.0],
    [1.0, 12.0, 1.0],
])
threshold = 0.2

variance_threshold_select(X, threshold)
Output:

python
Copy
1
2
3
4
5
⌄
array([
    [10.0, 0.0],
    [11.0, 0.0],
    [12.0, 1.0],
])


In [ ]:
import numpy as np

def variance_threshold_select(X, threshold):
    """
    Removes low-variance features (columns) from a dataset.

    Variance is defined as:
    Var(x) = (1 / n) * sum((x_i - mu)^2)

    Args:
        X (np.ndarray): 2D dataset of shape (n_samples, n_features).
        threshold (float): Keep only features with variance
            strictly greater than this value.

    Returns:
        np.ndarray: The transformed dataset with only selected
            columns, shape (n_samples, n_selected_features).
    """
    # Calculate variance along the columns (axis=0)
    variances = np.var(X, axis=0)

    # Boolean mask for features with variance > threshold
    mask = variances > threshold

    # Apply mask to select columns
    return X[:, mask]

result = variance_threshold_select(X, threshold)
result

# Q6. 94. Lag features




Build a lagged feature matrix from a 1D time series — turns past values into input features for ML models.

$$X_{t,j} = s_{t - \ell_j}, \quad y_t = s_t$$

where $s$ is the series and $\ell_j$ is the $j$-th lag.

**Constraints**
- No pandas — NumPy and built-in Python only
- Use lags in the order given — do not sort
- Only create rows where all lagged values exist → start at `t = max(lags)`

**Input**
```
series = [10.0, 11.0, 13.0, 12.0, 15.0]
lags   = [1, 3]
```
**Expected Output**
```
X = [[13.0, 10.0],   ← t=3: lag1=series[2]=13, lag3=series[0]=10
     [12.0, 11.0]]   ← t=4: lag1=series[3]=12, lag3=series[1]=11

y = [12.0, 15.0]     ← series[3], series[4]
```

In [ ]:
import numpy as np


def make_lag_features(series, lags):
    """
    Builds lag features for a 1D time series.

    Args:
        series (np.ndarray): Time-ordered values, length n.
        lags (list[int]): Positive lag steps (e.g., [1, 2, 7]).

    Returns:
        tuple[np.ndarray, np.ndarray]: (X, y) where:
          - X is a 2D array of rows, each row has len(lags) floats
          - y is a 1D array of targets aligned with X
    """
    n = len(series)
    max_lag = max(lags)

    # Number of valid rows in the feature matrix
    num_rows = n - max_lag

    # Initialize feature matrix and target vector
    x = np.empty((num_rows, len(lags)), dtype=float)
    y = series[max_lag:]

    # Fill feature matrix using provided lag order
    for j, lag in enumerate(lags):
        x[:, j] = series[max_lag - lag:n - lag]

    return x, y

result = make_lag_features(series, lags)
result

# Q7. 537. Feature correlation filtering




Remove redundant features whose absolute Pearson correlation with any already-kept feature exceeds a threshold.

$$\rho(x, y) = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2} \cdot \sqrt{\sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

**Constraints**
- No pandas, no sklearn — NumPy and built-in Python only
- Use absolute correlation: `abs(ρ) > threshold` → drop
- Greedy left-to-right: keep a feature only if not too correlated with any previously kept feature
- Return kept feature names in original order

**Input**
```
X = [[1.0, 2.0, 10.0],
     [2.0, 4.0,  9.0],
     [3.0, 6.0,  8.0],
     [4.0, 8.0,  7.0]]

feature_names = ["a", "b", "c"]
threshold     = 0.95
```
**Expected Output**
```
["a", "c"]

"a" kept    ← first feature, always kept
"b" dropped ← corr("a", "b") = 1.0  → abs = 1.0 > 0.95 → drop
"c" kept    ← corr("a", "c") ≈ -1.0 → abs = 1.0 > 0.95 → should drop too
              but problem expects ["a","c"] — verify threshold in your test
```

In [ ]:
import numpy as np


def filter_correlated_features(X, feature_names, threshold=0.9):
    """
    Removes highly correlated features using a greedy left-to-right rule.

    Args:
        X (np.ndarray): Data matrix of shape (n_samples, n_features).
        feature_names (list[str]): Names for each feature column, length n_features.
        threshold (float): Drop a feature if abs(corr) > threshold with any
            kept feature.

    Returns:
        list[str]: Names of the kept features in their original order.
    """
    # Compute feature-feature correlation matrix
    # rowvar=False means columns are variables (features)
    corr = np.corrcoef(X, rowvar=False)

    kept_indices = []
    kept_names = []

    # Greedy left-to-right selection
    for i, name in enumerate(feature_names):
        keep = True
        for j in kept_indices:
            if abs(corr[i, j]) > threshold:
                keep = False
                break
        if keep:
            kept_indices.append(i)
            kept_names.append(name)

    return kept_names

result = filter_correlated_features(X, feature_names, threshold)
result